# Customer Churn: EDA

This notebook is part of a multi-notebook customer churn project.

The goal of this notebook is to clean the Telco Customer Churn dataset and explore the major patterns associated with churn. The cleaned dataset produced here is used by the later insights, modeling, and recommendations notebooks.


## Project Notebooks

1. **Customer Churn: EDA**  
   Cleans the dataset, creates churn-related features, and explores major churn patterns.

2. **Customer Churn: Insights**  
   Builds customer segments and identifies high-risk churn groups.

3. **Customer Churn: Modeling**  
   Trains machine learning models to predict customer churn.

4. **Customer Churn: Recommendations**  
   Interprets segment and model results and turns findings into retention recommendations.


## Notebook Goals

In this notebook, we will:

- Load the raw Telco Customer Churn dataset
- Clean data quality issues, including `TotalCharges`
- Create churn-related helper columns
- Explore major churn patterns across customer, service, contract, and billing features
- Save a cleaned dataset for the later notebooks


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [ ]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
file_path = "/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
df.isna().sum()

In [ ]:
df_clean = df.copy()

df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

print("Missing values after converting TotalCharges:")
df_clean.isna().sum()[df_clean.isna().sum() > 0]

In [ ]:
df_clean[df_clean["TotalCharges"].isna()]

In [ ]:
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(0)

print("Remaining missing values:", df_clean.isna().sum().sum())

## Exploratory Data Analysis

The first goal is to understand the overall churn rate and how churn differs across customer groups.

In [ ]:
df_clean["Churn_Flag"] = df_clean["Churn"].map({"Yes": 1, "No": 0})

df_clean[["Churn", "Churn_Flag"]].head()

In [ ]:
churn_counts = df_clean["Churn"].value_counts()
churn_rate = df_clean["Churn_Flag"].mean()

print(churn_counts)
print(f"\nOverall churn rate: {churn_rate:.2%}")

In [ ]:
churn_counts.plot(kind="bar")
plt.title("Customer Churn Counts")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()

In [ ]:
contract_churn = (
    df_clean
    .groupby("Contract")["Churn_Flag"]
    .agg(Customer_Count="count", Churn_Rate="mean")
    .sort_values("Churn_Rate", ascending=False)
)

contract_churn["Churn_Rate_%"] = contract_churn["Churn_Rate"] * 100

contract_churn[["Customer_Count", "Churn_Rate_%"]]

In [ ]:
contract_churn["Churn_Rate_%"].plot(kind="bar")

plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=45)
plt.show()

## Churn by Tenure

Next, we analyze whether newer customers are more likely to churn than long-term customers.

In [ ]:
df_clean["Tenure_Group"] = pd.cut(
    df_clean["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12 months", "13-24 months", "25-48 months", "49-72 months"]
)

tenure_churn = (
    df_clean
    .groupby("Tenure_Group", observed=False)["Churn_Flag"]
    .agg(Customer_Count="count", Churn_Rate="mean")
)

tenure_churn["Churn_Rate_%"] = tenure_churn["Churn_Rate"] * 100

tenure_churn[["Customer_Count", "Churn_Rate_%"]]

In [ ]:
tenure_churn["Churn_Rate_%"].plot(kind="bar")

plt.title("Churn Rate by Tenure Group")
plt.xlabel("Tenure Group")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=45)
plt.show()

## Churn by Monthly Charges

This section checks whether customers with higher monthly charges are more likely to churn.

In [ ]:
df_clean.groupby("Churn")["MonthlyCharges"].describe()

In [ ]:
df_clean.boxplot(column="MonthlyCharges", by="Churn")

plt.title("Monthly Charges by Churn Status")
plt.suptitle("")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.show()

In [ ]:
internet_churn = (
    df_clean
    .groupby("InternetService")["Churn_Flag"]
    .agg(Customer_Count="count", Churn_Rate="mean")
    .sort_values("Churn_Rate", ascending=False)
)

internet_churn["Churn_Rate_%"] = internet_churn["Churn_Rate"] * 100

internet_churn[["Customer_Count", "Churn_Rate_%"]]

In [ ]:
internet_churn["Churn_Rate_%"].plot(kind="bar")

plt.title("Churn Rate by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=45)
plt.show()

In [ ]:
payment_churn = (
    df_clean
    .groupby("PaymentMethod")["Churn_Flag"]
    .agg(Customer_Count="count", Churn_Rate="mean")
    .sort_values("Churn_Rate", ascending=False)
)

payment_churn["Churn_Rate_%"] = payment_churn["Churn_Rate"] * 100

payment_churn[["Customer_Count", "Churn_Rate_%"]]

In [ ]:
payment_churn["Churn_Rate_%"].plot(kind="bar")

plt.title("Churn Rate by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.show()

## Initial EDA Findings

The initial analysis shows that churn is strongly related to contract type, customer tenure, monthly charges, internet service, and payment method.

Customers on month-to-month contracts have the highest churn rate, while customers on two-year contracts have the lowest churn rate. This suggests that longer contracts may be associated with stronger customer retention.

Newer customers are also more likely to churn. Customers with 0-12 months of tenure have a much higher churn rate than long-term customers, which suggests that the early customer experience may be important for retention.

Churned customers also have higher average monthly charges than non-churned customers. This may indicate that price sensitivity or perceived value plays a role in churn.

Fiber optic customers and customers using electronic checks show especially high churn rates. These groups may require deeper analysis because they represent potential high-risk customer segments.

In [ ]:
def churn_summary(column):
    summary = (
        df_clean
        .groupby(column)["Churn_Flag"]
        .agg(Customer_Count="count", Churn_Rate="mean")
        .sort_values("Churn_Rate", ascending=False)
    )

    summary["Churn_Rate_%"] = summary["Churn_Rate"] * 100

    return summary[["Customer_Count", "Churn_Rate_%"]]

In [ ]:
churn_summary("OnlineSecurity")

In [ ]:
churn_summary("TechSupport")

In [ ]:
churn_summary("OnlineBackup")

In [ ]:
churn_summary("DeviceProtection")

In [ ]:
churn_summary("PaperlessBilling")

In [ ]:
df_clean["Risk_Factor_Count"] = 0

df_clean["Risk_Factor_Count"] += (df_clean["Contract"] == "Month-to-month").astype(int)
df_clean["Risk_Factor_Count"] += (df_clean["tenure"] <= 12).astype(int)
df_clean["Risk_Factor_Count"] += (df_clean["InternetService"] == "Fiber optic").astype(int)
df_clean["Risk_Factor_Count"] += (df_clean["PaymentMethod"] == "Electronic check").astype(int)
df_clean["Risk_Factor_Count"] += (df_clean["MonthlyCharges"] > df_clean["MonthlyCharges"].median()).astype(int)

risk_churn = (
    df_clean
    .groupby("Risk_Factor_Count")["Churn_Flag"]
    .agg(Customer_Count="count", Churn_Rate="mean")
)

risk_churn["Churn_Rate_%"] = risk_churn["Churn_Rate"] * 100

risk_churn[["Customer_Count", "Churn_Rate_%"]]

In [ ]:
risk_churn["Churn_Rate_%"].plot(kind="bar")

plt.title("Churn Rate by Number of Risk Factors")
plt.xlabel("Number of Risk Factors")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=0)
plt.show()

## Service Feature Findings

Service-related features show clear churn patterns. Customers without online security, tech support, online backup, or device protection have noticeably higher churn rates than customers who have those services.

The strongest service-related churn signals are online security and tech support. Customers without online security churn at about 41.77%, compared with about 14.61% for customers with online security. Customers without tech support churn at about 41.64%, compared with about 15.17% for customers with tech support.

This suggests that customers who receive more support, protection, or account value from the company may be more likely to stay.

## Risk Factor Analysis

A simple risk factor score was created using five churn-related conditions:

- Month-to-month contract
- Tenure of 12 months or less
- Fiber optic internet service
- Electronic check payment method
- Monthly charges above the median

The churn rate increases sharply as the number of risk factors increases. Customers with no risk factors churn at only 2.54%, while customers with all five risk factors churn at 71.85%.

This does not replace a machine learning model, but it provides a clear business-friendly way to explain churn risk. It shows that churn is not driven by one single variable. Instead, customers become much more likely to churn when multiple risk factors appear together.

## EDA Conclusion

This notebook cleaned the Telco Customer Churn dataset and explored the main factors associated with customer churn.

The overall churn rate is 26.54%. The analysis found that churn is especially high among month-to-month customers, newer customers, fiber optic customers, electronic check users, customers with higher monthly charges, and customers without support or protection services.

The most important early findings are:

1. Month-to-month customers have much higher churn than one-year or two-year contract customers.
2. New customers are at greater risk, especially those with 0-12 months of tenure.
3. Churned customers tend to have higher monthly charges.
4. Fiber optic customers and electronic check users show high churn rates.
5. Customers without online security or tech support are more likely to churn.
6. Churn risk increases sharply when multiple risk factors are combined.

These findings will guide the next stage of the project, where customer segments and business recommendations will be developed before building machine learning models to predict churn.

In [ ]:
df_clean.to_csv("telco_churn_cleaned.csv", index=False)

print("Cleaned dataset saved as telco_churn_cleaned.csv")
print("Final shape:", df_clean.shape)